# 19 Tests of averages, spreads, shapes and order on real data, with the packages' own examples

The test words of the gap list (stream A, "Statistical tests", first half), run on the data of their parent
packages' own manual examples and on real datasets: `compare_average_to`, `compare_averages_of`,
`compare_before_and_after`, `sdata.compare_averages`, `compare_spreads_of`, `compare_shapes_of`,
`check_if_random_order`, `compare_shape_to`.

**Only simplydata runs in this notebook.** Every reference value (R functions simplydata does not call, formulas
written out in SciPy and NumPy, the numbers printed in SciPy docstrings and R help pages, Rouder et al.'s JZS
integral, exact enumerations) is computed by `scripts/references/19_tests_averages_spreads_shapes.py` and saved
to `references/19_tests_averages_spreads_shapes.csv`, with its source, tolerance and the package versions. The
data the words run on (R datasets, the data of docstring and help-page examples) are written by the same script
to `references/data/19_tests_averages_spreads_shapes/`; the notebook reads them with `sdata.table`.

**Two kinds of evidence**: the manual examples (McDonald's mussel shells for SciPy `f_oneway`, Darwin's maize
for `wilcoxon`, Brunner & Munzel's pain scores, Mood's median example, the house prices of `alexandergovern`,
the `levene` and `mannwhitneyu` examples; Hollander & Wolfe's serum iron for R `ansari.test` and `mood.test`,
their depression scores for `wilcox.test`, Agresti's presidential approval for `mcnemar.test`, Brownlee's density
of the earth for randtests `runs.test`, TOSTER's mtcars and sleep examples, the kSamples `ad.test` example), and
independent references on real data (Student's sleep data, Michelson's speed of light, PlantGrowth,
InsectSprays, warpbreaks, CO2 uptake, the wind directions of R circular).

Tolerances (stored with each reference): closed formulas 1e-10 relative, docstring values 1e-10, exact
permutation counts 1e-12, values integrated numerically or fitted by iteration 1e-6, Monte Carlo 4 Monte Carlo
standard errors.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_row, read_rows, versions

NOTEBOOK = "19_tests_averages_spreads_shapes"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def answer(word, *args, **options):
    """The one-row answer of a simplydata word, without its printout, read with table.get_row(1)."""
    return read_row(quiet(word, *args, **options))


def label(word, **options):
    return word + (" (" + ", ".join(f"{k}={v!r}" for k, v in options.items()) + ")" if options else "")

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## `compare_average_to`: Michelson's speed of light (R `morley`) and Darwin's maize (the `wilcoxon` docstring)

Michelson's 100 measurements of 1879 (km/s minus 299 000) against today's value, 792.458 in those units:
the t test with both one-sided forms, the z test with a known standard deviation (80), the two one-sided tests
within 50 km/s and the two non-inferiority forms.

In [2]:
light = load("morley")
light.compare_average_to("Speed", 792.458)

The average "Speed" is 852, which is 59.9 higher than 792.458 (95 % CI of the average 837 to 868; Cohen's d = 0.759 (95 % CI 0.535 to 0.980)).
If the average were truly 792.458, a difference this big or bigger would turn up in fewer than 1 in 1000 studies like this one (p < 0.0001).
Test: one-sample t-test (t = 7.59, df = 99), the usual test for one average against a fixed number.


,test,what it compares,statistic name,statistic,degrees of freedom,p-value,effect size name,effect size,effect size 95 % CI low,effect size 95 % CI high,average,average 95 % CI low,average 95 % CI high,rows
1,one-sample t-test,"the average of ""Speed"" against 792.458",t,7.586582,99.0,1.823745e-11,difference in standard deviations (Cohen's d),0.758658,0.534551,0.979759,852.4,836.722593,868.077407,100


In [3]:
DS = "morley (Michelson 1879)"
for options in [{}, {"sides": 1, "higher": "average"}, {"sides": 1, "lower": "average"}, {"alternative": "less"}]:
    checks.check_values(DS, label("compare_average_to", **options), answer(light.compare_average_to, "Speed", 792.458, **options))
checks.check_values(DS, 'compare_average_to (test_method="z", standard_deviation=)',
                    answer(light.compare_average_to, "Speed", 792.458, test_method="z", standard_deviation=80))
checks.check_values(DS, "compare_average_to (same_within=)", answer(light.compare_average_to, "Speed", 792.458, same_within=50))
for option in ("not_lower", "not_higher"):
    found = answer(light.compare_average_to, "Speed", 792.458, same_within=50, **{option: "average"})
    checks.check(DS, f'compare_average_to (same_within=, {option}="average")', "p-value", found["p-value"])

agrees  morley (Michelson 1879) | compare_average_to | statistic: simplydata 7.58658200134, R t.test 7.58658200134; relative difference 0.00e+00 (allowed 1e-10)
      Source: R t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  morley (Michelson 1879) | compare_average_to | degrees of freedom: simplydata 99, R t.test 99; relative difference 0.00e+00 (allowed 1e-10)
      Source: R t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  morley (Michelson 1879) | compare_average_to | p-value: simplydata 1.82374451271e-11, R t.test 1.82374451271e-11; relative difference 5.67e-15 (allowed 1e-10)
      Source: R t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  morley (Michelson 1879) | compare_average_to | effect size: simplydata 0.758658200134, R t.test 0.758658200134; relative difference 0.00e+00 (allowed 1e-10)
      Source: R t.test (run reference) - https://s

Darwin's maize: the height differences of 15 pairs of cross- and self-fertilised plants, from the SciPy
`wilcoxon` docstring, whose printed p-values (0.041259765625 two-sided; 0.0206298828125 and the statistic 96
for `alternative='greater'`) are references beside R `wilcox.test`. Then the Bayes factor with
`prior_width="wide"` and `0.5`, and the posterior with `with_seed=7`.

In [4]:
maize = load("darwin")
ds = "Darwin's maize (SciPy wilcoxon docstring)"
checks.check_values(ds, 'compare_average_to (based_on="ranks")', answer(maize.compare_average_to, "difference", 0, based_on="ranks"))
found = answer(maize.compare_average_to, "difference", 0, based_on="ranks", sides=1, higher="average")
checks.check_values(ds, 'compare_average_to (based_on="ranks", sides=1, higher="average")',
                    {"p-value": found["p-value"], "statistic (V)": found["statistic"]})
for options in [{"sides": 1, "higher": "average"}, {"alternative": "less"}]:
    checks.check_values(ds, 'compare_average_to (based_on="ranks"' + "".join(f", {k}={v!r}" for k, v in options.items()) + ")",
                        answer(maize.compare_average_to, "difference", 0, based_on="ranks", **options))
for width in ("wide", 0.5):
    found = answer(maize.compare_average_to, "difference", 0, test_method="bayesian", prior_width=width, with_seed=7)
    checks.check(ds, f'compare_average_to (test_method="bayesian", prior_width={width!r})', "Bayes factor", found["Bayes factor"])
    checks.check_values(ds, f'compare_average_to (test_method="bayesian", prior_width={width!r}, with_seed=7)', found)

agrees  Darwin's maize (SciPy wilcoxon docstring) | compare_average_to (based_on="ranks") | statistic: simplydata 96, R wilcox.test (exact) 96; relative difference 0.00e+00 (allowed 1e-10)
      Source: R wilcox.test (exact) (run reference) - https://search.r-project.org/R/refmans/stats/html/wilcox.test.html
agrees  Darwin's maize (SciPy wilcoxon docstring) | compare_average_to (based_on="ranks") | p-value: simplydata 0.041259765625, SciPy wilcoxon docstring, printed 0.041259765625; relative difference 0.00e+00 (allowed 1e-10)
      Source: SciPy wilcoxon docstring, printed (printed) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.wilcoxon.html
agrees  Darwin's maize (SciPy wilcoxon docstring) | compare_average_to (based_on="ranks") | p-value: simplydata 0.041259765625, R wilcox.test (exact) 0.041259765625; relative difference 0.00e+00 (allowed 1e-10)
      Source: R wilcox.test (exact) (run reference) - https://search.r-project.org/R/refmans/stats/html/wilcox

agrees  Darwin's maize (SciPy wilcoxon docstring) | compare_average_to (test_method="bayesian", prior_width='wide') | Bayes factor: simplydata 1.31088552833, Rouder et al. (2009) integral, SciPy quad 1.31088552833; relative difference 4.18e-14 (allowed 1e-06)
      Source: Rouder et al. (2009) integral, SciPy quad (run reference)
agrees  Darwin's maize (SciPy wilcoxon docstring) | compare_average_to (test_method="bayesian", prior_width='wide') | Bayes factor: simplydata 1.31088552833, R BayesFactor ttestBF 1.31088552833; relative difference 1.69e-16 (allowed 1e-06)
      Source: R BayesFactor ttestBF (run reference) - https://search.r-project.org/CRAN/refmans/BayesFactor/html/ttestBF.html (version 0.9.12.4.8)
agrees  Darwin's maize (SciPy wilcoxon docstring) | compare_average_to (test_method="bayesian", prior_width='wide', with_seed=7) | difference: simplydata 19.0240382655, R BayesFactor posterior, same seed 19.0240382655; relative difference 1.87e-16 (allowed 1e-10)
      Source: R B

## `compare_averages_of`: the docstring examples of `f_oneway`, `median_test`, `alexandergovern`, `mannwhitneyu`
and `brunnermunzel`

In [5]:
shells = load("mussels")
shells.compare_averages_of("length", for_each_group_in="place", test_method="anova")

The average "length" is 0.0780 in magadan, 0.0748 in newport, 0.103 in petersburg, 0.0802 in tillamook and 0.0957 in tvarminne. Omega-squared = 0.386 (95 % CI 0.0868 to 0.560): the groups explain about 39 % of the differences in "length".
If the groups were truly alike, differences this big or bigger between them would turn up in fewer than 1 in 1000 studies like this one (p = 0.0003).
Test: one-way ANOVA (F = 7.12, df = 4 and 34), chosen with test_method=; it assumes the groups have the same spread. To see which groups differ: table.compare_each_pair_of("length", for_each_group_in="place").
Note: the values in group "magadan" don't look bell-shaped (Shapiro-Wilk p = 0.036); a safer test here: based_on="ranks".
Note: the values in group "tvarminne" don't look bell-shaped (Shapiro-Wilk p = 0.013); a safer test here: based_on="ranks".


,test,what it compares,statistic name,statistic,degrees of freedom 1,degrees of freedom 2,p-value,effect size name,effect size,effect size 95 % CI low,...,count in magadan,average in magadan,count in newport,average in newport,count in petersburg,average in petersburg,count in tillamook,average in tillamook,count in tvarminne,average in tvarminne
1,one-way ANOVA,"the average of ""length"" in each ""place"" (magad...",F,7.121019,4.0,34.0,0.000281,share of the differences explained by the grou...,0.385673,0.086819,...,8,0.078012,8,0.0748,7,0.103443,10,0.0802,6,0.0957


In [6]:
ds = "mussel shells (SciPy f_oneway docstring; McDonald et al. 1991)"
for test in ("anova", "welch anova", "kruskal-wallis", "median"):
    checks.check_values(ds, f'compare_averages_of (test_method="{test}")',
                        answer(shells.compare_averages_of, "length", for_each_group_in="place", test_method=test))
checks.check_values("SciPy median_test docstring", 'compare_averages_of (test_method="median")',
                    answer(load("groups3").compare_averages_of, "y", for_each_group_in="group", test_method="median"))
checks.check_values("house prices (SciPy alexandergovern docstring)", 'compare_averages_of (test_method="alexander-govern")',
                    answer(load("houses").compare_averages_of, "price", for_each_group_in="city", test_method="alexander-govern"))
found = answer(load("ages").compare_averages_of, "age", for_each_group_in="sex", test_method="mann-whitney")
checks.check_values("ages (SciPy mannwhitneyu docstring)", 'compare_averages_of (test_method="mann-whitney")',
                    {**found, "statistic (U of men)": found["statistic"]})
checks.check_values("pain scores (SciPy brunnermunzel docstring; Brunner & Munzel 2000)",
                    'compare_averages_of (test_method="brunner-munzel")',
                    answer(load("pain").compare_averages_of, "score", for_each_group_in="group", test_method="brunner-munzel"))

agrees  mussel shells (SciPy f_oneway docstring; McDonald et al. 1991) | compare_averages_of (test_method="anova") | statistic: simplydata 7.12101947164, SciPy f_oneway docstring, printed 7.12101947164; relative difference 2.49e-16 (allowed 1e-10)
      Source: SciPy f_oneway docstring, printed (printed) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.f_oneway.html
agrees  mussel shells (SciPy f_oneway docstring; McDonald et al. 1991) | compare_averages_of (test_method="anova") | statistic: simplydata 7.12101947164, R anova(lm()) 7.12101947164; relative difference 1.25e-16 (allowed 1e-10)
      Source: R anova(lm()) (run reference) - https://search.r-project.org/R/refmans/stats/html/anova.html
agrees  mussel shells (SciPy f_oneway docstring; McDonald et al. 1991) | compare_averages_of (test_method="anova") | p-value: simplydata 0.000281224231453, SciPy f_oneway docstring, printed 0.000281224231453; relative difference 1.93e-13 (allowed 1e-10)
      Source: Sci

agrees  mussel shells (SciPy f_oneway docstring; McDonald et al. 1991) | compare_averages_of (test_method="kruskal-wallis") | statistic: simplydata 16.4048105891, R kruskal.test 16.4048105891; relative difference 0.00e+00 (allowed 1e-10)
      Source: R kruskal.test (run reference) - https://search.r-project.org/R/refmans/stats/html/kruskal.test.html
agrees  mussel shells (SciPy f_oneway docstring; McDonald et al. 1991) | compare_averages_of (test_method="kruskal-wallis") | p-value: simplydata 0.00252140145376, R kruskal.test 0.00252140145376; relative difference 3.78e-15 (allowed 1e-10)
      Source: R kruskal.test (run reference) - https://search.r-project.org/R/refmans/stats/html/kruskal.test.html
agrees  mussel shells (SciPy f_oneway docstring; McDonald et al. 1991) | compare_averages_of (test_method="median") | statistic: simplydata 10.6194360902, R chisq.test on above/not above 10.6194360902; relative difference 0.00e+00 (allowed 1e-10)
      Source: R chisq.test on above/not abo

4

## `compare_averages_of` on Student's sleep data: Student and Welch t, one-sided tests, Yuen, shuffling,
equivalence (TOSTER's mtcars example) and the Bayes factor

With 200 000 shuffles simplydata counts every one of the 184 756 splits, so its p-value is exact (against R perm
`permTS(method = "exact.network")`).

In [7]:
sleepers = load("sleep")
ds = "sleep (Cushny & Peebles 1905), as two groups"
for test in ("welch", "student"):
    for options in [{}, {"sides": 1, "higher": "drug 2"}, {"sides": 1, "lower": "drug 2"}, {"alternative": "less"},
                    {"alternative": "two-sided"}]:
        checks.check_values(ds, f'compare_averages_of (test_method="{test}"' + "".join(f", {k}={v!r}" for k, v in options.items()) + ")",
                            answer(sleepers.compare_averages_of, "extra", for_each_group_in="drug", test_method=test, **options))
found = answer(sleepers.compare_averages_of, "extra", for_each_group_in="drug", test_method="yuen")
found["statistic (size)"] = abs(found["statistic"])   # R WRS2 reports the size of the statistic
checks.check_values(ds, 'compare_averages_of (test_method="yuen")', found)
found = answer(sleepers.compare_averages_of, "extra", for_each_group_in="drug", p_value_by="shuffling", shuffles=9999, with_seed=1)
checks.check(ds, 'compare_averages_of (p_value_by="shuffling", shuffles=9999, with_seed=1)', "p-value", found["p-value"])
found = answer(sleepers.compare_averages_of, "extra", for_each_group_in="drug", p_value_by="shuffling", shuffles=200000)
checks.check(ds, 'compare_averages_of (p_value_by="shuffling", shuffles=200000: every split)', "p-value", found["p-value"])
for width in ("wide", 0.5):
    found = answer(sleepers.compare_averages_of, "extra", for_each_group_in="drug", test_method="bayesian",
                   prior_width=width, with_seed=7)
    checks.check(ds, f'compare_averages_of (test_method="bayesian", prior_width={width!r})', "Bayes factor", found["Bayes factor"])
    checks.check_values(ds, f'compare_averages_of (test_method="bayesian", prior_width={width!r}, with_seed=7)', found)

cars = load("mtcars")
ds = "mtcars mpg by transmission (TOSTER t_TOST example 1)"
checks.check_values(ds, "compare_averages_of (same_within=)",
                    answer(cars.compare_averages_of, "mpg", for_each_group_in="transmission", same_within=3))
for option in ("not_lower", "not_higher"):
    found = answer(cars.compare_averages_of, "mpg", for_each_group_in="transmission", same_within=3, **{option: "manual"})
    checks.check(ds, f'compare_averages_of (same_within=, {option}="manual")', "p-value", found["p-value"])

agrees  sleep (Cushny & Peebles 1905), as two groups | compare_averages_of (test_method="welch") | statistic: simplydata -1.86081346749, R t.test -1.86081346749; relative difference 1.19e-16 (allowed 1e-10)
      Source: R t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  sleep (Cushny & Peebles 1905), as two groups | compare_averages_of (test_method="welch") | degrees of freedom: simplydata 17.7764735162, R t.test 17.7764735162; relative difference 6.00e-16 (allowed 1e-10)
      Source: R t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  sleep (Cushny & Peebles 1905), as two groups | compare_averages_of (test_method="welch") | p-value: simplydata 0.0793941401874, R t.test 0.0793941401874; relative difference 2.45e-15 (allowed 1e-10)
      Source: R t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  sleep (Cushny & Peebles 1905), as two groups | compare_

agrees  sleep (Cushny & Peebles 1905), as two groups | compare_averages_of (p_value_by="shuffling", shuffles=9999, with_seed=1) | p-value: simplydata 0.0834, SciPy permutation_test, same seed 0.0834; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy permutation_test, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.permutation_test.html


agrees  sleep (Cushny & Peebles 1905), as two groups | compare_averages_of (p_value_by="shuffling", shuffles=200000: every split) | p-value: simplydata 0.0814479638009, R perm permTS (exact) 0.0814479638009; relative difference 1.02e-15 (allowed 1e-12)
      Source: R perm permTS (exact) (run reference) - https://search.r-project.org/CRAN/refmans/perm/html/permTS.html (version 1.0.0.4)
agrees  sleep (Cushny & Peebles 1905), as two groups | compare_averages_of (test_method="bayesian", prior_width='wide') | Bayes factor: simplydata 1.14793805975, Rouder et al. (2009) integral, SciPy quad 1.14793805987; relative difference 9.93e-11 (allowed 1e-06)
      Source: Rouder et al. (2009) integral, SciPy quad (run reference)
agrees  sleep (Cushny & Peebles 1905), as two groups | compare_averages_of (test_method="bayesian", prior_width='wide') | Bayes factor: simplydata 1.14793805975, R BayesFactor ttestBF 1.14793805975; relative difference 0.00e+00 (allowed 1e-06)
      Source: R BayesFactor tte

Ordered groups (Jonckheere-Terpstra on warpbreaks, against R PMCMRplus) and repeated measures (`matched_by=`:
the CO2 uptake of 12 plants at 7 concentrations, against base R `aov` with `Error()` and Greenhouse and
Geisser's epsilon from the covariance matrix).

In [8]:
found = answer(load("warpbreaks").compare_averages_of, "breaks", for_each_group_in="tension", test_method="jonckheere",
               in_order=["L", "M", "H"])
checks.check("warpbreaks (Tippett 1950)", 'compare_averages_of (test_method="jonckheere", in_order=)', "p-value", found["p-value"])
found = answer(load("co2").compare_averages_of, "uptake", for_each_group_in="conc", matched_by="Plant")
print(found)
gg = [c for c in found if "p-value" in c and ("Greenhouse" in c or "corrected" in c)]
epsilon = [c for c in found if "epsilon" in c.lower()]
values = {"statistic (F)": found["statistic"], "p-value (Greenhouse-Geisser)": found[gg[0] if gg else "p-value"]}
if epsilon:
    values["Greenhouse-Geisser epsilon"] = found[epsilon[0]]
checks.check_values("CO2 uptake (Potvin et al. 1990)", 'compare_averages_of (matched_by="Plant")', values)

agrees  warpbreaks (Tippett 1950) | compare_averages_of (test_method="jonckheere", in_order=) | p-value: simplydata 0.00082364091211, R PMCMRplus jonckheereTest 0.00082364091211; relative difference 1.10e-13 (allowed 1e-10)
      Source: R PMCMRplus jonckheereTest (run reference) - https://search.r-project.org/CRAN/refmans/PMCMRplus/html/jonckheereTest.html (version 1.9.12)


{'term': 'conc', 'statistic name': 'F', 'statistic': 57.67630836751378, 'degrees of freedom 1': 1.4294182552678776, 'degrees of freedom 2': 15.723600807946655, 'p-value': 2.5057393386493983e-07, 'generalised eta-squared': 0.4191595403379224, 'Greenhouse-Geisser epsilon': 0.2382363758779796, 'Mauchly W': 0.00010032471959970858, 'Mauchly p-value': 8.521487063914926e-09}
agrees  CO2 uptake (Potvin et al. 1990) | compare_averages_of (matched_by="Plant") | statistic (F): simplydata 57.6763083675, R aov with Error() 57.6763083675; relative difference 1.23e-16 (allowed 1e-10)
      Source: R aov with Error() (run reference) - https://search.r-project.org/R/refmans/stats/html/aov.html
agrees  CO2 uptake (Potvin et al. 1990) | compare_averages_of (matched_by="Plant") | p-value (Greenhouse-Geisser): simplydata 2.50573933865e-07, epsilon from the covariance matrix, R pf 2.50573933865e-07; relative difference 9.93e-15 (allowed 1e-10)
      Source: epsilon from the covariance matrix, R pf (run refe

3

## `compare_before_and_after`

Hollander & Wolfe's depression scores (R `wilcox.test` help page; "before" is the first visit), Student's sleep
data as the paired design it is (paired t, sign test, all 1024 sign patterns for the shuffled p-value, TOSTER's
second example with the bounds -0.5 and 2, the Bayes factor), and Agresti's presidential approval (R
`mcnemar.test` help page; McNemar, and the exact p-value).

In [9]:
visits = load("depression")
ds = "depression scores (R wilcox.test example; Hollander & Wolfe 1973)"
for options in [{"sides": 1, "higher": "before"}, {}, {"sides": 1, "lower": "before"}, {"alternative": "less"}]:
    found = answer(visits.compare_before_and_after, "first", "second", based_on="ranks", **options)
    checks.check(ds, 'compare_before_and_after (based_on="ranks"' + "".join(f", {k}={v!r}" for k, v in options.items()) + ")",
                 "p-value", found["p-value"])

patients = load("paired")
ds = "sleep (Cushny & Peebles 1905), paired"
for options in [{}, {"sides": 1, "higher": "after"}, {"sides": 1, "lower": "after"}, {"alternative": "greater"}]:
    checks.check_values(ds, label("compare_before_and_after", **options),
                        answer(patients.compare_before_and_after, "drug1", "drug2", **options))
checks.check_values(ds, 'compare_before_and_after (test_method="sign")',
                    answer(patients.compare_before_and_after, "drug1", "drug2", test_method="sign"))
found = answer(patients.compare_before_and_after, "drug1", "drug2", p_value_by="shuffling", shuffles=9999, with_seed=1)
checks.check(ds, 'compare_before_and_after (p_value_by="shuffling", shuffles=, with_seed=)', "p-value", found["p-value"])
for width in ("wide", 0.5):
    found = answer(patients.compare_before_and_after, "drug1", "drug2", test_method="bayesian", prior_width=width, with_seed=7)
    checks.check(ds, f'compare_before_and_after (test_method="bayesian", prior_width={width!r})', "Bayes factor",
                 found["Bayes factor"])
ds = "sleep, paired (TOSTER t_TOST example 2)"   # TOSTER tests drug1 - drug2: before = drug2, after = drug1
checks.check_values(ds, "compare_before_and_after (same_within=)",
                    answer(patients.compare_before_and_after, "drug2", "drug1", same_within=[-0.5, 2]))
for option in ("not_lower", "not_higher"):
    found = answer(patients.compare_before_and_after, "drug2", "drug1", same_within=[-0.5, 2], **{option: "after"})
    checks.check(ds, f'compare_before_and_after (same_within=, {option}="after")', "p-value", found["p-value"])

surveys = load("approval")
ds = "presidential approval (R mcnemar.test example; Agresti 1990)"
checks.check_values(ds, "compare_before_and_after (yes/no columns: McNemar)",
                    answer(surveys.compare_before_and_after, "first survey", "second survey"))
found = answer(surveys.compare_before_and_after, "first survey", "second survey", p_value_by="exact")
checks.check(ds, 'compare_before_and_after (p_value_by="exact")', "p-value", found["p-value"])

agrees  depression scores (R wilcox.test example; Hollander & Wolfe 1973) | compare_before_and_after (based_on="ranks", sides=1, higher='before') | p-value: simplydata 0.01953125, R wilcox.test (exact) 0.01953125; relative difference 0.00e+00 (allowed 1e-10)
      Source: R wilcox.test (exact) (run reference) - https://search.r-project.org/R/refmans/stats/html/wilcox.test.html
agrees  depression scores (R wilcox.test example; Hollander & Wolfe 1973) | compare_before_and_after (based_on="ranks") | p-value: simplydata 0.0390625, R wilcox.test (exact) 0.0390625; relative difference 0.00e+00 (allowed 1e-10)
      Source: R wilcox.test (exact) (run reference) - https://search.r-project.org/R/refmans/stats/html/wilcox.test.html
agrees  depression scores (R wilcox.test example; Hollander & Wolfe 1973) | compare_before_and_after (based_on="ranks", sides=1, lower='before') | p-value: simplydata 0.986328125, R wilcox.test (exact) 0.986328125; relative difference 0.00e+00 (allowed 1e-10)
      So

agrees  sleep (Cushny & Peebles 1905), paired | compare_before_and_after (test_method="bayesian", prior_width=0.5) | Bayes factor: simplydata 15.0304133086, Rouder et al. (2009) integral, SciPy quad 15.0304133086; relative difference 1.08e-12 (allowed 1e-06)
      Source: Rouder et al. (2009) integral, SciPy quad (run reference)
agrees  sleep, paired (TOSTER t_TOST example 2) | compare_before_and_after (same_within=) | p-value against the low margin: simplydata 0.989240756622, paired TOST written out in R 0.989240756622; relative difference 1.12e-16 (allowed 1e-10)
      Source: paired TOST written out in R (run reference)
agrees  sleep, paired (TOSTER t_TOST example 2) | compare_before_and_after (same_within=) | p-value against the low margin: simplydata 0.989240756622, R TOSTER t_TOST (help page example) 0.989240756622; relative difference 1.12e-16 (allowed 1e-10)
      Source: R TOSTER t_TOST (help page example) (run reference) - https://search.r-project.org/CRAN/refmans/TOSTER/html

agrees  sleep, paired (TOSTER t_TOST example 2) | compare_before_and_after (same_within=, not_lower="after") | p-value: simplydata 0.989240756622, one-sided paired t written out in R 0.989240756622; relative difference 1.12e-16 (allowed 1e-10)
      Source: one-sided paired t written out in R (run reference)
agrees  sleep, paired (TOSTER t_TOST example 2) | compare_before_and_after (same_within=, not_higher="after") | p-value: simplydata 3.55234415925e-06, one-sided paired t written out in R 3.55234415925e-06; relative difference 5.96e-16 (allowed 1e-10)
      Source: one-sided paired t written out in R (run reference)
agrees  presidential approval (R mcnemar.test example; Agresti 1990) | compare_before_and_after (yes/no columns: McNemar) | statistic: simplydata 16.8177966102, R mcnemar.test 16.8177966102; relative difference 0.00e+00 (allowed 1e-10)
      Source: R mcnemar.test (run reference) - https://search.r-project.org/R/refmans/stats/html/mcnemar.test.html
agrees  presidential a

agrees  presidential approval (R mcnemar.test example; Agresti 1990) | compare_before_and_after (p_value_by="exact") | p-value: simplydata 3.71593613957e-05, R binom.test on the changes 3.71593613957e-05; relative difference 1.17e-14 (allowed 1e-10)
      Source: R binom.test on the changes (run reference) - https://search.r-project.org/R/refmans/stats/html/binom.test.html


1

## `sdata.compare_averages`: from the summary numbers of the three PlantGrowth groups

The averages, standard deviations and counts are taken from the rows with simplydata
(`find_summary_of`), as a paper would print them; the references are R `t.test` and `oneway.test` on the rows.

In [10]:
plants = load("plants")
summary = read_rows(quiet(plants.find_summary_of, "weight", for_each_group_in="group",
                          show=["average", "standard deviation", "count"]))
by_group = {row["group"]: row for row in summary}
ds = "PlantGrowth (Dobson 1983), from summary numbers"
for groups, test, method in [(["ctrl", "trt1"], None, "sdata.compare_averages"),
                             (["ctrl", "trt1"], "student", 'sdata.compare_averages (test_method="student")'),
                             (["ctrl", "trt1", "trt2"], None, "sdata.compare_averages (3 groups)"),
                             (["ctrl", "trt1", "trt2"], "anova", 'sdata.compare_averages (test_method="anova")')]:
    found = answer(sdata.compare_averages, averages=[by_group[g]["average"] for g in groups],
                   standard_deviations=[by_group[g]["standard deviation"] for g in groups],
                   counts=[by_group[g]["count"] for g in groups], test_method=test)
    checks.check_values(ds, method, found)

agrees  PlantGrowth (Dobson 1983), from summary numbers | sdata.compare_averages | statistic: simplydata 1.19126038185, R t.test on the rows 1.19126038185; relative difference 0.00e+00 (allowed 1e-10)
      Source: R t.test on the rows (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  PlantGrowth (Dobson 1983), from summary numbers | sdata.compare_averages | degrees of freedom: simplydata 16.5235850569, R t.test on the rows 16.5235850569; relative difference 0.00e+00 (allowed 1e-10)
      Source: R t.test on the rows (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  PlantGrowth (Dobson 1983), from summary numbers | sdata.compare_averages | p-value: simplydata 0.250382508588, R t.test on the rows 0.250382508588; relative difference 0.00e+00 (allowed 1e-10)
      Source: R t.test on the rows (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  PlantGrowth (Dobson 1983), from sum

## `compare_spreads_of`, `compare_shapes_of` and `check_if_random_order`

In [11]:
serum = load("serum")
ds = "serum iron (R ansari.test / mood.test example; Hollander & Wolfe 1973)"
for test in ("ansari", "mood"):
    found = answer(serum.compare_spreads_of, "iron", for_each_group_in="method", test_method=test)
    checks.check(ds, f'compare_spreads_of (test_method="{test}")', "p-value", found["p-value"])
found = answer(serum.compare_spreads_of, "iron", for_each_group_in="method", test_method="f")
checks.check_values(ds, 'compare_spreads_of (test_method="f")', {"statistic (F)": found["statistic"], "p-value": found["p-value"]})
for name, table, column, group in [("SciPy levene docstring", load("lev"), "y", "group"),
                                   ("PlantGrowth (Dobson 1983)", plants, "weight", "group")]:
    for test in ("levene", "brown-forsythe"):
        checks.check_values(name, f'compare_spreads_of (test_method="{test}")',
                            answer(table.compare_spreads_of, column, for_each_group_in=group, test_method=test))
insects = load("insects")
for test in ("bartlett", "fligner"):
    checks.check_values("InsectSprays (Beall 1942)", f'compare_spreads_of (test_method="{test}")',
                        answer(insects.compare_spreads_of, "count", for_each_group_in="spray", test_method=test))

two = load("plants")
two.keep_rows_where("group", is_one_of=["ctrl", "trt1"])
checks.check_values("PlantGrowth ctrl, trt1", "compare_shapes_of", answer(two.compare_shapes_of, "weight", for_each_group_in="group"))
checks.check_values("serum iron (Hollander & Wolfe)", "compare_shapes_of", answer(serum.compare_shapes_of, "iron", for_each_group_in="method"))
found = answer(load("three").compare_shapes_of, "value", for_each_group_in="sample", shuffles=9999, with_seed=1)
checks.check("R kSamples ad.test example", "compare_shapes_of (3 groups, shuffles=9999, with_seed=1)", "p-value", found["p-value"])
checks.check("R kSamples ad.test example", "compare_shapes_of (3 groups)", "statistic (standardised, version 2)", found["statistic"])

for name, table, column in [("density of the earth (R randtests runs.test example; Brownlee 1965)", load("earth"), "density"),
                            ("sweet potato yield 1868-1937 (R randtests)", load("potatoes"), "yield")]:
    checks.check_values(name, "check_if_random_order", answer(table.check_if_random_order, column))

agrees  serum iron (R ansari.test / mood.test example; Hollander & Wolfe 1973) | compare_spreads_of (test_method="ansari") | p-value: simplydata 0.181458199729, R ansari.test 0.181458199729; relative difference 0.00e+00 (allowed 1e-10)
      Source: R ansari.test (run reference) - https://search.r-project.org/R/refmans/stats/html/ansari.test.html
agrees  serum iron (R ansari.test / mood.test example; Hollander & Wolfe 1973) | compare_spreads_of (test_method="mood") | p-value: simplydata 0.307757612978, R mood.test 0.307757612978; relative difference 1.62e-15 (allowed 1e-10)
      Source: R mood.test (run reference) - https://search.r-project.org/R/refmans/stats/html/mood.test.html
agrees  serum iron (R ansari.test / mood.test example; Hollander & Wolfe 1973) | compare_spreads_of (test_method="f") | statistic (F): simplydata 0.715064198268, variance ratio, NumPy 0.715064198268; relative difference 6.21e-16 (allowed 1e-10)
      Source: variance ratio, NumPy (run reference)
agrees  serum

agrees  R kSamples ad.test example | compare_shapes_of (3 groups, shuffles=9999, with_seed=1) | p-value: simplydata 0.0409, SciPy anderson_ksamp, same seed 0.0409; relative difference 2.38e-15 (allowed 1e-12)
      Source: SciPy anderson_ksamp, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.anderson_ksamp.html
agrees  R kSamples ad.test example | compare_shapes_of (3 groups, shuffles=9999, with_seed=1) | p-value: simplydata 0.0409, R kSamples ad.test, help page run (1000 draws; 4 MCSE) 0.03; difference 1.09e-02 (allowed 2e-02)
      Source: R kSamples ad.test, help page run (1000 draws; 4 MCSE) (run reference) - https://search.r-project.org/CRAN/refmans/kSamples/html/ad.test.html (version 1.2.12)
agrees  R kSamples ad.test example | compare_shapes_of (3 groups, shuffles=9999, with_seed=1) | p-value: simplydata 0.0409, R kSamples ad.test, 200 000 draws (4 MCSE) 0.038425; difference 2.47e-03 (allowed 8e-03)
      Source: R kSamples ad.

## `compare_shape_to`

Normality tests on Michelson's speeds and Brownlee's densities; against a fixed normal distribution (the average
of Darwin's differences and 1.1 times their standard deviation, both found with simplydata), the
Kolmogorov-Smirnov and Cramer-von Mises tests; a fitted gamma with its simulated p-value; the Rayleigh test on
the wind directions (radians, and degrees with `full_turn=360`).

In [12]:
for name, table, column in [("morley (Michelson 1879)", light, "Speed"), ("density of the earth (Brownlee 1965)", load("earth"), "density")]:
    checks.check_values(name, "compare_shape_to (Shapiro-Wilk)", answer(table.compare_shape_to, column))
    for test in ("lilliefors", "dagostino", "jarque-bera"):
        checks.check_values(name, f'compare_shape_to (test_method="{test}")', answer(table.compare_shape_to, column, test_method=test))

differences = maize.get_column("difference")
fixed = sdata.distribution("normal", average=quiet(differences.find_average),
                           standard_deviation=1.1 * quiet(differences.find_standard_deviation))
ds = "Darwin's maize (SciPy wilcoxon docstring)"
checks.check_values(ds, "compare_shape_to (shape=a fixed distribution: Kolmogorov-Smirnov)", answer(maize.compare_shape_to, "difference", fixed))
checks.check_values(ds, 'compare_shape_to (test_method="cramer-von-mises")',
                    answer(maize.compare_shape_to, "difference", fixed, test_method="cramer-von-mises"))

potatoes = load("potatoes")
ds = "sweet potato yield (R randtests)"
found = answer(potatoes.compare_shape_to, "yield", "gamma", samples=9999, with_seed=1)
checks.check(ds, 'compare_shape_to (shape="gamma", samples=9999, with_seed=1)', "p-value, run again", found["p-value"])
checks.check(ds, 'compare_shape_to (shape="gamma")', "statistic (Anderson-Darling)", found["statistic"])
fewer = answer(potatoes.compare_shape_to, "yield", "gamma", samples=999, with_seed=2)
checks.check(ds, 'compare_shape_to (shape="gamma", samples=)', "p-value, 999 against 9999 samples", fewer["p-value"])

wind = load("wind")
for column, turn, options in [("d", "radians", {"shape": "even around the circle"}),
                              ("degrees", 360, {"shape": "even around the circle"}),
                              ("degrees", 360, {"shape": "uniform"}),
                              ("degrees", 360, {"shape": "even around the circle", "test_method": "rayleigh"})]:
    found = answer(wind.compare_shape_to, column, full_turn=turn, **options)
    checks.check("wind directions (R circular)",
                 "compare_shape_to (" + ", ".join(f"{k}={v!r}" for k, v in options.items()) + f", full_turn={turn!r})",
                 "p-value", found["p-value"])

agrees  morley (Michelson 1879) | compare_shape_to (Shapiro-Wilk) | statistic: simplydata 0.988074329913, R shapiro.test 0.988074329913; relative difference 3.37e-16 (allowed 1e-10)
      Source: R shapiro.test (run reference) - https://search.r-project.org/R/refmans/stats/html/shapiro.test.html
agrees  morley (Michelson 1879) | compare_shape_to (Shapiro-Wilk) | p-value: simplydata 0.513703926147, R shapiro.test 0.513703926148; relative difference 6.70e-14 (allowed 1e-10)
      Source: R shapiro.test (run reference) - https://search.r-project.org/R/refmans/stats/html/shapiro.test.html
agrees  morley (Michelson 1879) | compare_shape_to (test_method="lilliefors") | statistic: simplydata 0.083424374274, largest gap, NumPy 0.083424374274; relative difference 4.99e-16 (allowed 1e-10)
      Source: largest gap, NumPy (run reference)
agrees  morley (Michelson 1879) | compare_shape_to (test_method="lilliefors") | statistic: simplydata 0.083424374274, R nortest lillie.test 0.083424374274; relat

agrees  sweet potato yield (R randtests) | compare_shape_to (shape="gamma", samples=9999, with_seed=1) | p-value, run again: simplydata 0.1405, simplydata, same seed 0.1405; difference 0.00e+00 (allowed 0e+00)
      Source: simplydata, same seed (run reference)
agrees  sweet potato yield (R randtests) | compare_shape_to (shape="gamma") | statistic (Anderson-Darling): simplydata 0.575698889198, R goftest ad.test at simplydata's fit 0.575698889198; relative difference 2.45e-14 (allowed 1e-10)
      Source: R goftest ad.test at simplydata's fit (run reference) - https://search.r-project.org/CRAN/refmans/goftest/html/ad.test.html (version 1.2.3)
agrees  sweet potato yield (R randtests) | compare_shape_to (shape="gamma", samples=) | p-value, 999 against 9999 samples: simplydata 0.13, simplydata 9999 samples (4 MCSE) 0.1405; difference 1.05e-02 (allowed 5e-02)
      Source: simplydata 9999 samples (4 MCSE) (run reference)
agrees  wind directions (R circular) | compare_shape_to (shape='even a

## Result

In [13]:
checks.finish()

19_tests_averages_spreads_shapes: 244 comparisons - 244 agrees
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,19_tests_averages_spreads_shapes,morley (Michelson 1879),compare_average_to,statistic,7.586582e+00,7.586582e+00,R t.test,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1,19_tests_averages_spreads_shapes,morley (Michelson 1879),compare_average_to,degrees of freedom,9.900000e+01,9.900000e+01,R t.test,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
2,19_tests_averages_spreads_shapes,morley (Michelson 1879),compare_average_to,p-value,1.823745e-11,1.823745e-11,R t.test,-1.033976e-25,1.000000e-10,relative,5.669521e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3,19_tests_averages_spreads_shapes,morley (Michelson 1879),compare_average_to,effect size,7.586582e-01,7.586582e-01,R t.test,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
4,19_tests_averages_spreads_shapes,morley (Michelson 1879),compare_average_to,average 95 % CI low,8.367226e+02,8.367226e+02,R t.test,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,19_tests_averages_spreads_shapes,wind directions (R circular),compare_shape_to (shape='even around the circl...,p-value,1.293979e-58,1.293979e-58,R circular rayleigh.test,-1.768687e-74,1.000000e-10,relative,1.366860e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/circ...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
240,19_tests_averages_spreads_shapes,wind directions (R circular),"compare_shape_to (shape='uniform', full_turn=360)",p-value,1.293979e-58,1.293979e-58,Rayleigh test written out in R,-1.768687e-74,1.000000e-10,relative,1.366860e-16,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
241,19_tests_averages_spreads_shapes,wind directions (R circular),"compare_shape_to (shape='uniform', full_turn=360)",p-value,1.293979e-58,1.293979e-58,R circular rayleigh.test,-1.768687e-74,1.000000e-10,relative,1.366860e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/circ...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
242,19_tests_averages_spreads_shapes,wind directions (R circular),compare_shape_to (shape='even around the circl...,p-value,1.293979e-58,1.293979e-58,Rayleigh test written out in R,-1.768687e-74,1.000000e-10,relative,1.366860e-16,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
